In [1]:
# Cell 1: نصب پکیج‌های مورد نیاز
!pip install -q transformer-lens torch numpy accelerate

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 5.8 MB/s eta 0:00:00


In [2]:
"""
m10_1_transferability_predictor.py
===================================
Milestone 10.1: Structural Transferability Predictor.

Question this script answers:
  "Before running an expensive intervention experiment on a NEW component,
   can we predict — from structural properties alone (layer distance,
   head vs MLP, upstream vs downstream) — whether our existing
   micro-fingerprint Phi(x) will actually be predictive for it?"

This is deliberately NOT a claim of a trained, generalizable classifier.
With only ~10 target components, any fitted model would be unverifiable.
Instead, this script:
  1. Runs the SAME rigorous per-target audit as M9.4 (LOO-Ridge + 500-perm
     test + FDR correction) across 10 STRUCTURALLY DIVERSE targets
     (same-layer heads, same-layer MLP, downstream heads/MLP, upstream
     heads/MLP) instead of just 4.
  2. Reports, per target, both the predictability result AND its
     structural descriptors (layer distance, is_mlp).
  3. Computes a simple correlation between structural distance and
     predictability as an EXPLORATORY, hypothesis-generating signal —
     explicitly labeled as such, not as a validated predictor.

Reuses the exact harness/probe/statistics pattern from M9.4 so results
are directly comparable.
"""

import torch
import numpy as np
import json
from dataclasses import dataclass
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


# ==============================================================================
# 1. TARGET COMPONENT SUITE (STRUCTURALLY DIVERSE, N=10)
# ==============================================================================

@dataclass
class TargetComponent:
    name: str
    layer: int
    is_mlp: bool
    head: int = 0        # ignored if is_mlp
    scale: float = 0.0   # intervention strength (0.0 = zero-ablate)

    def descriptor(self, base_layer: int = 21) -> Dict[str, float]:
        return {
            "layer_distance": float(self.layer - base_layer),   # signed: + = downstream
            "abs_layer_distance": float(abs(self.layer - base_layer)),
            "is_mlp": 1.0 if self.is_mlp else 0.0,
            "same_layer": 1.0 if self.layer == base_layer else 0.0,
        }


def get_target_suite() -> List[TargetComponent]:
    return [
        # Same layer, different heads
        TargetComponent("L21H02_Ablate", layer=21, is_mlp=False, head=2, scale=0.0),
        TargetComponent("L21H09_Ablate", layer=21, is_mlp=False, head=9, scale=0.0),
        # Same layer, MLP
        TargetComponent("L21MLP_Damp", layer=21, is_mlp=True, scale=0.5),
        # Downstream heads (near and far)
        TargetComponent("L22H05_Ablate", layer=22, is_mlp=False, head=5, scale=0.0),
        TargetComponent("L23H01_Ablate", layer=23, is_mlp=False, head=1, scale=0.0),
        # Downstream MLP
        TargetComponent("L22MLP_Damp", layer=22, is_mlp=True, scale=0.5),
        # Upstream heads (near and far)
        TargetComponent("L18H03_Boost", layer=18, is_mlp=False, head=3, scale=2.0),
        TargetComponent("L14H04_Ablate", layer=14, is_mlp=False, head=4, scale=0.0),
        TargetComponent("L10H00_Ablate", layer=10, is_mlp=False, head=0, scale=0.0),
        # Upstream MLP
        TargetComponent("L18MLP_Damp", layer=18, is_mlp=True, scale=0.5),
    ]


# ==============================================================================
# 2. MECHANISTIC HARNESS (same probe family as M9.4 / M9.5)
# ==============================================================================

class TransferabilityHarness:
    def __init__(self, model_name: str = "Qwen/Qwen2.5-0.5B", device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"🔄 Loading {model_name} on {self.device.upper()} (bfloat16)...")
        self.model = HookedTransformer.from_pretrained(
            model_name, device=self.device, torch_dtype=torch.bfloat16, default_padding_side="left"
        )
        self.model.eval()

    @torch.no_grad()
    def extract_phi_and_all_targets(
        self, prompt: str, targets: List[TargetComponent]
    ) -> Tuple[np.ndarray, Dict[str, float]]:
        tokens = self.model.to_tokens(prompt)
        nom_logits = self.model(tokens)[0, -1, :].detach().cpu().to(torch.float32)

        top_ids = torch.topk(nom_logits, k=2).indices
        t1, t2 = int(top_ids[0].item()), int(top_ids[1].item())
        base_margin = float((nom_logits[t1] - nom_logits[t2]).item())

        # Fixed 4D micro-fingerprint from the KNOWN base circuit (L21H06, L21H01, L20MLP)
        def p1_h6(act, hook): act[:, -1, 6, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p1_h6)]):
            d1 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        def p2_h6(act, hook): act[:, -1, 6, :] *= 1.10; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p2_h6)]):
            d2 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        def p3_h1(act, hook): act[:, -1, 1, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p3_h1)]):
            d3 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        def p4_mlp20(act, hook): act[:, -1, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.20.hook_mlp_out", p4_mlp20)]):
            d4 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        phi = np.array([d1, d2, d3, d4], dtype=np.float32)

        # Ground-truth effect on each of the 10 structurally diverse targets
        target_deltas = {}
        for tgt in targets:
            if tgt.is_mlp:
                hook_name = f"blocks.{tgt.layer}.hook_mlp_out"
                def hook_fn(act, hook, sc=tgt.scale): act[:, -1, :] *= sc; return act
            else:
                hook_name = f"blocks.{tgt.layer}.attn.hook_z"
                def hook_fn(act, hook, h=tgt.head, sc=tgt.scale): act[:, -1, h, :] *= sc; return act

            with self.model.hooks(fwd_hooks=[(hook_name, hook_fn)]):
                m_logits = self.model(tokens)[0, -1, :].detach().cpu().to(torch.float32)
            target_deltas[tgt.name] = float((m_logits[t1] - m_logits[t2]).item()) - base_margin

        return phi, target_deltas


# ==============================================================================
# 3. EVALUATION CORPUS (48 balanced tasks — facts / bindings / distractors)
# ==============================================================================

def get_balanced_corpus() -> List[Dict]:
    return [
        {"id": "F01", "prompt": "The capital of France is the city of"},
        {"id": "F02", "prompt": "The capital of Japan is the bustling city of"},
        {"id": "F03", "prompt": "The official capital city of Germany is"},
        {"id": "F04", "prompt": "The historic capital of Italy is the city of"},
        {"id": "F05", "prompt": "The capital of Spain is the city of"},
        {"id": "F06", "prompt": "The capital of England is the city of"},
        {"id": "F07", "prompt": "The capital of Russia is the city of"},
        {"id": "F08", "prompt": "The capital of Egypt is the city of"},
        {"id": "F09", "prompt": "The capital of Greece is the city of"},
        {"id": "F10", "prompt": "The capital of China is the city of"},
        {"id": "F11", "prompt": "The capital of Canada is the city of"},
        {"id": "F12", "prompt": "The capital of Australia is the city of"},
        {"id": "B01", "prompt": "Alice handed a red apple to Bob. The apple was received by"},
        {"id": "B02", "prompt": "David passed the heavy book to Sarah. The book was taken by"},
        {"id": "B03", "prompt": "John sent a private email to Mary. The email reached"},
        {"id": "B04", "prompt": "Michael offered a cup of tea to Emma. The tea was accepted by"},
        {"id": "B05", "prompt": "Tom tossed the baseball to Jack. The ball was caught by"},
        {"id": "B06", "prompt": "Robert wrote a letter to Lisa. The letter was delivered to"},
        {"id": "B07", "prompt": "William awarded a shiny medal to Oliver. The medal went to"},
        {"id": "B08", "prompt": "James threw the yellow frisbee to Lucas. The frisbee was caught by"},
        {"id": "B09", "prompt": "Daniel gave a fresh sandwich to Mason. The sandwich was eaten by"},
        {"id": "B10", "prompt": "Henry presented an award to Ethan. The award was received by"},
        {"id": "B11", "prompt": "Samuel mailed a package to Alexander. The package arrived for"},
        {"id": "B12", "prompt": "George loaned some money to Noah. The money helped"},
        {"id": "D01", "prompt": "While Rome has the Colosseum, the Eiffel Tower stands in"},
        {"id": "D02", "prompt": "Even though Madrid is famous, the capital of Germany is"},
        {"id": "D03", "prompt": "Berlin has monuments, yet the ancient Colosseum stands in"},
        {"id": "D04", "prompt": "Although Tokyo is very large, the ancient Parthenon is located in"},
        {"id": "D05", "prompt": "Despite London being historic, the capital of Spain is definitely"},
        {"id": "D06", "prompt": "Paris is beautiful, but the capital of Italy is definitely"},
        {"id": "D07", "prompt": "Egypt has pyramids, but the Eiffel Tower was built in the city of"},
        {"id": "D08", "prompt": "Although Moscow has the Kremlin, the capital of France is the city of"},
    ]


# ==============================================================================
# 4. STATISTICS (identical methodology to M9.3A / M9.4 for direct comparability)
# ==============================================================================

def leave_one_out_ridge(X: np.ndarray, y: np.ndarray, alpha: float = 0.5) -> Tuple[float, float]:
    N, D = X.shape
    preds = []
    for i in range(N):
        tr_idx = [j for j in range(N) if j != i]
        X_tr, y_tr = X[tr_idx], y[tr_idx]
        w = np.linalg.solve(X_tr.T @ X_tr + alpha * np.eye(D), X_tr.T @ y_tr)
        preds.append(float(np.dot(X[i], w)))
    preds = np.array(preds)
    mae = float(np.mean(np.abs(preds - y)))
    corr = float(np.corrcoef(preds, y)[0, 1]) if np.std(preds) > 1e-6 else 0.0
    return mae, corr


def benjamini_hochberg(p_values: List[float], alpha: float = 0.05) -> Tuple[List[bool], float]:
    p_arr = np.array(p_values)
    n = len(p_arr)
    sorted_idx = np.argsort(p_arr)
    sorted_p = p_arr[sorted_idx]
    thresholds = (np.arange(1, n + 1) / n) * alpha
    sig_mask = sorted_p <= thresholds
    p_crit = float(sorted_p[np.max(np.where(sig_mask)[0])]) if np.any(sig_mask) else 0.0
    sig_flags = (p_arr <= p_crit).tolist()
    return sig_flags, p_crit


# ==============================================================================
# 5. MASTER RUNNER
# ==============================================================================

def run_milestone_10_1():
    harness = TransferabilityHarness()
    corpus = get_balanced_corpus()
    targets = get_target_suite()

    print("\n" + "=" * 140)
    print(f"🔬 MILESTONE 10.1: STRUCTURAL TRANSFERABILITY AUDIT (N={len(corpus)} tasks x {len(targets)} target components)")
    print("=" * 140)

    phi_list = []
    target_records: Dict[str, List[float]] = {t.name: [] for t in targets}

    for item in corpus:
        phi, deltas = harness.extract_phi_and_all_targets(item["prompt"], targets)
        phi_list.append(phi)
        for t in targets:
            target_records[t.name].append(deltas[t.name])

    X_phi = np.stack(phi_list)
    print(f"✅ Extracted 4D fixed fingerprints (from base circuit L21H06/L21H01/L20MLP) "
          f"and ground-truth deltas for {len(targets)} target components.\n")

    # -------------------------------------------------------------------------
    # PER-TARGET AUDIT: identical methodology to M9.4 (LOO-Ridge + 500-perm + FDR)
    # -------------------------------------------------------------------------
    print("=" * 140)
    print("📊 PER-TARGET PREDICTABILITY AUDIT (500-Permutation Test)")
    print("=" * 140)
    print(f"{'Target':<18} | {'Layer Dist':<10} | {'Is MLP':<7} | {'Phi MAE':<10} | {'Phi ρ':<9} | "
          f"{'Global Mean MAE':<16} | {'p-value':<10} | {'FDR Sig'}")
    print("-" * 140)

    n_perms = 500
    raw_p_values = []
    audit_rows = []

    for tgt in targets:
        y = np.array(target_records[tgt.name])
        mae_phi, corr_phi = leave_one_out_ridge(X_phi, y)
        mae_mean = float(np.mean(np.abs(y - np.mean(y))))

        shuff_maes = []
        for _ in range(n_perms):
            y_s = np.random.permutation(y)
            m_s, _ = leave_one_out_ridge(X_phi, y_s)
            shuff_maes.append(m_s)
        p_val = float((np.sum(np.array(shuff_maes) <= mae_phi) + 1) / (n_perms + 1))
        raw_p_values.append(p_val)

        desc = tgt.descriptor(base_layer=21)
        audit_rows.append({
            "target": tgt.name,
            "descriptor": desc,
            "mae_phi": mae_phi,
            "corr_phi": corr_phi,
            "mae_mean": mae_mean,
            "p_value": p_val,
        })

    fdr_flags, p_crit = benjamini_hochberg(raw_p_values, alpha=0.05)

    for row, sig in zip(audit_rows, fdr_flags):
        d = row["descriptor"]
        sig_str = "🟢 SIGNIFICANT" if sig else "🔴 NOT SIGNIFICANT"
        print(f"{row['target']:<18} | {d['layer_distance']:+10.0f} | {'MLP' if d['is_mlp'] else 'HEAD':<7} | "
              f"{row['mae_phi']:8.4f}   | {row['corr_phi']:+7.4f}   | {row['mae_mean']:14.4f}   | "
              f"p={row['p_value']:.4f}  | {sig_str}")
        row["fdr_significant"] = bool(sig)

    # -------------------------------------------------------------------------
    # EXPLORATORY ANALYSIS: does structural distance predict transferability?
    # This is explicitly NOT a trained/validated classifier — n=10 is far too
    # small for that. It is a descriptive correlation intended to generate
    # hypotheses for which structural relationships to test next, at scale.
    # -------------------------------------------------------------------------
    print("\n" + "=" * 140)
    print("📊 EXPLORATORY: DOES STRUCTURAL DISTANCE CORRELATE WITH PREDICTABILITY? (n=10 — hypothesis-generating only)")
    print("=" * 140)

    abs_dist = np.array([row["descriptor"]["abs_layer_distance"] for row in audit_rows])
    is_mlp_arr = np.array([row["descriptor"]["is_mlp"] for row in audit_rows])
    corr_vals = np.array([row["corr_phi"] for row in audit_rows])

    if np.std(abs_dist) > 1e-6 and np.std(corr_vals) > 1e-6:
        dist_vs_corr = float(np.corrcoef(abs_dist, corr_vals)[0, 1])
    else:
        dist_vs_corr = float("nan")

    if np.std(is_mlp_arr) > 1e-6 and np.std(corr_vals) > 1e-6:
        mlp_vs_corr = float(np.corrcoef(is_mlp_arr, corr_vals)[0, 1])
    else:
        mlp_vs_corr = float("nan")

    print(f"  • corr(abs layer distance, predictability ρ) : {dist_vs_corr:+.3f}  (n=10, NOT a validated model)")
    print(f"  • corr(is_MLP, predictability ρ)             : {mlp_vs_corr:+.3f}  (n=10, NOT a validated model)")
    print("  ⚠️  With only 10 target components, treat these as directional hints to guide")
    print("     which structural relationships to sample MORE of in the next batch —")
    print("     not as a claim that structural distance 'explains' transferability.")

    sig_count = sum(fdr_flags)
    print("\n" + "-" * 140)
    print(f"Summary: {sig_count}/{len(targets)} targets showed FDR-significant transfer from the base-circuit fingerprint.")
    print("=" * 140 + "\n")

    with open("m10_1_transferability_results.json", "w", encoding="utf-8") as f:
        json.dump({
            "audit_rows": audit_rows,
            "fdr_p_critical": p_crit,
            "exploratory_correlations": {
                "abs_distance_vs_predictability": dist_vs_corr,
                "is_mlp_vs_predictability": mlp_vs_corr,
            },
            "n_significant": int(sig_count),
            "n_targets": len(targets),
        }, f, indent=2)
    print("📁 Telemetry saved to: m10_1_transferability_results.json")


if __name__ == "__main__":
    run_milestone_10_1()


🔄 Loading Qwen/Qwen2.5-0.5B on CUDA (bfloat16)...


/tmp/ipykernel_1228/3807724422.py:91: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

...

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer

🔬 MILESTONE 10.1: STRUCTURAL TRANSFERABILITY AUDIT (N=32 tasks x 10 target components)
✅ Extracted 4D fixed fingerprints (from base circuit L21H06/L21H01/L20MLP) and ground-truth deltas for 10 target components.

📊 PER-TARGET PREDICTABILITY AUDIT (500-Permutation Test)
Target             | Layer Dist | Is MLP  | Phi MAE    | Phi ρ     | Global Mean MAE  | p-value    | FDR Sig
--------------------------------------------------------------------------------------------------------------------------------------------
L21H02_Ablate      |         +0 | HEAD    |   0.0724   | +0.0464   |         0.0771   | p=0.7944  | 🔴 NOT SIGNIFICANT
L21H09_Ablate      |         +0 | HEAD    |   0.3058   | -0.0057   |         0.2832   | p=0.9621  | 🔴 NOT SIGNIFICANT
L21MLP_Damp        |         +0 | MLP     |   0.4117   | +0.6365   |         0.5684   | p=0.0020  | 🟢 SIGNIFICANT
L22H05_Ablate      |         +1 | HEAD    |   0.0698   | +0.5764

In [ ]:
"""
m10_2_l22_head_specificity_audit_fixed.py
==========================================
Milestone 10.2: L22H05 Intra-Layer Specificity & Circuit Replication Audit.

Fix applied:
  - Corrected head indices for Qwen2.5-0.5B (n_heads = 14, indices 0..13).
  - Target list now uses valid heads: H0, H2, H4, H5 (target), H7, H9, H11, H13.
"""

import torch
import numpy as np
import json
from dataclasses import dataclass
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


# ==============================================================================
# 1. LAYER 22 TARGET HEADS (TARGET + 7 INTRA-LAYER CONTROLS, 0..13)
# ==============================================================================

@dataclass
class HeadTarget:
    name: str
    layer: int
    head: int
    scale: float = 0.0  # Zero-ablation intervention


def get_l22_head_suite() -> List[HeadTarget]:
    return [
        HeadTarget("L22H00_Ablate", layer=22, head=0),
        HeadTarget("L22H02_Ablate", layer=22, head=2),
        HeadTarget("L22H04_Ablate", layer=22, head=4),
        HeadTarget("L22H05_Ablate", layer=22, head=5),  # Primary candidate circuit node
        HeadTarget("L22H07_Ablate", layer=22, head=7),
        HeadTarget("L22H09_Ablate", layer=22, head=9),
        HeadTarget("L22H11_Ablate", layer=22, head=11),
        HeadTarget("L22H13_Ablate", layer=22, head=13), # Corrected index (0..13)
    ]


# ==============================================================================
# 2. MECHANISTIC HARNESS
# ==============================================================================

class L22SpecificityHarness:
    def __init__(self, model_name: str = "Qwen/Qwen2.5-0.5B", device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"🔄 Loading {model_name} on {self.device.upper()} (bfloat16)...")
        self.model = HookedTransformer.from_pretrained(
            model_name, device=self.device, torch_dtype=torch.bfloat16, default_padding_side="left"
        )
        self.model.eval()

    @torch.no_grad()
    def extract_phi_and_l22_deltas(
        self, prompt: str, targets: List[HeadTarget]
    ) -> Tuple[np.ndarray, Dict[str, float]]:
        tokens = self.model.to_tokens(prompt)
        nom_logits = self.model(tokens)[0, -1, :].detach().cpu().to(torch.float32)

        top_ids = torch.topk(nom_logits, k=2).indices
        t1, t2 = int(top_ids[0].item()), int(top_ids[1].item())
        base_margin = float((nom_logits[t1] - nom_logits[t2]).item())

        # 4D Micro-Fingerprint (L21H06, L21H01, L20MLP)
        def p1_h6(act, hook): act[:, -1, 6, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p1_h6)]):
            d1 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        def p2_h6(act, hook): act[:, -1, 6, :] *= 1.10; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p2_h6)]):
            d2 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        def p3_h1(act, hook): act[:, -1, 1, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p3_h1)]):
            d3 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        def p4_mlp20(act, hook): act[:, -1, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.20.hook_mlp_out", p4_mlp20)]):
            d4 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        phi = np.array([d1, d2, d3, d4], dtype=np.float32)

        # Ground-truth intervention consequence on all L22 target heads
        target_deltas = {}
        for tgt in targets:
            def hook_fn(act, hook, h=tgt.head, sc=tgt.scale):
                act[:, -1, h, :] *= sc
                return act

            with self.model.hooks(fwd_hooks=[(f"blocks.{tgt.layer}.attn.hook_z", hook_fn)]):
                m_logits = self.model(tokens)[0, -1, :].detach().cpu().to(torch.float32)
            target_deltas[tgt.name] = float((m_logits[t1] - m_logits[t2]).item()) - base_margin

        return phi, target_deltas


# ==============================================================================
# 3. BALANCED EVALUATION CORPUS (N=32 TASKS)
# ==============================================================================

def get_balanced_corpus() -> List[Dict]:
    return [
        {"id": "F01", "prompt": "The capital of France is the city of"},
        {"id": "F02", "prompt": "The capital of Japan is the bustling city of"},
        {"id": "F03", "prompt": "The official capital city of Germany is"},
        {"id": "F04", "prompt": "The historic capital of Italy is the city of"},
        {"id": "F05", "prompt": "The capital of Spain is the city of"},
        {"id": "F06", "prompt": "The capital of England is the city of"},
        {"id": "F07", "prompt": "The capital of Russia is the city of"},
        {"id": "F08", "prompt": "The capital of Egypt is the city of"},
        {"id": "F09", "prompt": "The capital of Greece is the city of"},
        {"id": "F10", "prompt": "The capital of China is the city of"},
        {"id": "F11", "prompt": "The capital of Canada is the city of"},
        {"id": "F12", "prompt": "The capital of Australia is the city of"},
        {"id": "B01", "prompt": "Alice handed a red apple to Bob. The apple was received by"},
        {"id": "B02", "prompt": "David passed the heavy book to Sarah. The book was taken by"},
        {"id": "B03", "prompt": "John sent a private email to Mary. The email reached"},
        {"id": "B04", "prompt": "Michael offered a cup of tea to Emma. The tea was accepted by"},
        {"id": "B05", "prompt": "Tom tossed the baseball to Jack. The ball was caught by"},
        {"id": "B06", "prompt": "Robert wrote a letter to Lisa. The letter was delivered to"},
        {"id": "B07", "prompt": "William awarded a shiny medal to Oliver. The medal went to"},
        {"id": "B08", "prompt": "James threw the yellow frisbee to Lucas. The frisbee was caught by"},
        {"id": "B09", "prompt": "Daniel gave a fresh sandwich to Mason. The sandwich was eaten by"},
        {"id": "B10", "prompt": "Henry presented an award to Ethan. The award was received by"},
        {"id": "B11", "prompt": "Samuel mailed a package to Alexander. The package arrived for"},
        {"id": "B12", "prompt": "George loaned some money to Noah. The money helped"},
        {"id": "D01", "prompt": "While Rome has the Colosseum, the Eiffel Tower stands in"},
        {"id": "D02", "prompt": "Even though Madrid is famous, the capital of Germany is"},
        {"id": "D03", "prompt": "Berlin has monuments, yet the ancient Colosseum stands in"},
        {"id": "D04", "prompt": "Although Tokyo is very large, the ancient Parthenon is located in"},
        {"id": "D05", "prompt": "Despite London being historic, the capital of Spain is definitely"},
        {"id": "D06", "prompt": "Paris is beautiful, but the capital of Italy is definitely"},
        {"id": "D07", "prompt": "Egypt has pyramids, but the Eiffel Tower was built in the city of"},
        {"id": "D08", "prompt": "Although Moscow has the Kremlin, the capital of France is the city of"},
    ]


# ==============================================================================
# 4. STATISTICAL EVALUATION ENGINE
# ==============================================================================

def leave_one_out_ridge(X: np.ndarray, y: np.ndarray, alpha: float = 0.5) -> Tuple[float, float]:
    N, D = X.shape
    preds = []
    for i in range(N):
        tr_idx = [j for j in range(N) if j != i]
        X_tr, y_tr = X[tr_idx], y[tr_idx]
        w = np.linalg.solve(X_tr.T @ X_tr + alpha * np.eye(D), X_tr.T @ y_tr)
        preds.append(float(np.dot(X[i], w)))
    preds = np.array(preds)
    mae = float(np.mean(np.abs(preds - y)))
    corr = float(np.corrcoef(preds, y)[0, 1]) if np.std(preds) > 1e-6 else 0.0
    return mae, corr


def benjamini_hochberg(p_values: List[float], alpha: float = 0.05) -> Tuple[List[bool], float]:
    p_arr = np.array(p_values)
    n = len(p_arr)
    sorted_idx = np.argsort(p_arr)
    sorted_p = p_arr[sorted_idx]
    thresholds = (np.arange(1, n + 1) / n) * alpha
    sig_mask = sorted_p <= thresholds
    p_crit = float(sorted_p[np.max(np.where(sig_mask)[0])]) if np.any(sig_mask) else 0.0
    sig_flags = (p_arr <= p_crit).tolist()
    return sig_flags, p_crit


# ==============================================================================
# 5. MASTER EXECUTION ENGINE
# ==============================================================================

def run_milestone_10_2():
    harness = L22SpecificityHarness()
    corpus = get_balanced_corpus()
    targets = get_l22_head_suite()

    print("\n" + "=" * 140)
    print(f"🔬 MILESTONE 10.2: L22H05 INTRA-LAYER SPECIFICITY AUDIT (N={len(corpus)} Tasks x {len(targets)} Heads in Layer 22)")
    print("=" * 140)

    phi_list = []
    target_records: Dict[str, List[float]] = {t.name: [] for t in targets}

    for item in corpus:
        phi, deltas = harness.extract_phi_and_l22_deltas(item["prompt"], targets)
        phi_list.append(phi)
        for t in targets:
            target_records[t.name].append(deltas[t.name])

    X_phi = np.stack(phi_list)
    print(f"✅ Extracted 4D fingerprints and measured 8 Layer-22 heads across all {len(corpus)} episodes.\n")

    # -------------------------------------------------------------------------
    # STATISTICAL EVALUATION & 500-PERMUTATION AUDIT
    # -------------------------------------------------------------------------
    print("=" * 140)
    print("📊 INTRA-LAYER PREDICTABILITY SPECTRUM (Layer 22 Heads)")
    print("=" * 140)
    print(f"{'Target Head':<18} | {'Phi MAE':<10} | {'Phi ρ':<10} | {'Global Mean MAE':<16} | {'Perm p-val':<12} | {'FDR Sig (q < 0.05)'}")
    print("-" * 140)

    n_perms = 500
    raw_p_values = []
    audit_rows = []

    for tgt in targets:
        y = np.array(target_records[tgt.name])
        mae_phi, corr_phi = leave_one_out_ridge(X_phi, y)
        mae_mean = float(np.mean(np.abs(y - np.mean(y))))

        shuff_maes = []
        for _ in range(n_perms):
            y_s = np.random.permutation(y)
            m_s, _ = leave_one_out_ridge(X_phi, y_s)
            shuff_maes.append(m_s)
        p_val = float((np.sum(np.array(shuff_maes) <= mae_phi) + 1) / (n_perms + 1))
        raw_p_values.append(p_val)

        audit_rows.append({
            "target": tgt.name,
            "head": tgt.head,
            "mae_phi": mae_phi,
            "corr_phi": corr_phi,
            "mae_mean": mae_mean,
            "p_value": p_val,
        })

    fdr_flags, p_crit = benjamini_hochberg(raw_p_values, alpha=0.05)

    for row, sig in zip(audit_rows, fdr_flags):
        sig_str = "🟢 SIGNIFICANT" if sig else "🔴 NOT SIGNIFICANT"
        is_target_marker = " ◄ [Candidate Circuit Node]" if row["head"] == 5 else ""
        print(f"{row['target']:<18} | {row['mae_phi']:8.4f}   | {row['corr_phi']:+8.4f}   | {row['mae_mean']:14.4f}   | "
              f"p = {row['p_value']:.4f}     | {sig_str}{is_target_marker}")
        row["fdr_significant"] = bool(sig)

    # -------------------------------------------------------------------------
    # SPECIFICITY & OUTLIER METRICS
    # -------------------------------------------------------------------------
    print("\n" + "=" * 140)
    print("📊 SPECIFICITY & CIRCUIT CONNECTIVITY ANALYSIS")
    print("=" * 140)

    control_corrs = [r["corr_phi"] for r in audit_rows if r["head"] != 5]
    h5_corr = next(r["corr_phi"] for r in audit_rows if r["head"] == 5)

    mean_ctrl = float(np.mean(control_corrs))
    std_ctrl = float(np.std(control_corrs)) if np.std(control_corrs) > 1e-6 else 1e-4
    z_score_h5 = (h5_corr - mean_ctrl) / std_ctrl

    print(f"  • L22H05 Correlation (ρ)               : {h5_corr:+.4f}")
    print(f"  • Control Heads Mean Correlation (μ)    : {mean_ctrl:+.4f} (± {std_ctrl:.4f})")
    print(f"  • Specificity Z-Score vs. Layer 22      : Z = {z_score_h5:+.2f}")
    print("-" * 140)

    h5_is_sig = next(r["fdr_significant"] for r in audit_rows if r["head"] == 5)
    n_other_sig = sum(1 for r in audit_rows if r["head"] != 5 and r["fdr_significant"])

    if h5_is_sig and z_score_h5 >= 1.50 and n_other_sig <= 2:
        print("👉 FINAL AUDIT VERDICT: 🟢 L22H05_PRIVILEGED_CIRCUIT_NODE_VALIDATED")
        print("   (L22H05 is an isolated, highly-predictable downstream circuit node; specificity confirmed)")
    elif h5_is_sig and n_other_sig > 2:
        print("👉 FINAL AUDIT VERDICT: 🟡 LAYER_WIDE_DOWNSTREAM_TRANSFER")
        print("   (L22H05 is predictable, but transferability is broadly distributed across Layer 22 heads)")
    else:
        print("👉 FINAL AUDIT VERDICT: 🔴 INSUFFICIENT_SPECIFICITY")
    print("=" * 140 + "\n")

    with open("m10_2_l22_specificity_results.json", "w", encoding="utf-8") as f:
        json.dump({
            "audit_rows": audit_rows,
            "fdr_p_critical": p_crit,
            "metrics": {
                "h5_corr": h5_corr,
                "control_heads_mean_corr": mean_ctrl,
                "control_heads_std_corr": std_ctrl,
                "z_score_h5": z_score_h5,
                "n_significant_total": sum(fdr_flags),
            }
        }, f, indent=2)
    print("📁 Comprehensive Telemetry saved to: m10_2_l22_specificity_results.json")


if __name__ == "__main__":
    run_milestone_10_2()

🔄 Loading Qwen/Qwen2.5-0.5B on CUDA (bfloat16)...


/tmp/ipykernel_1228/3025040914.py:56: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer

🔬 MILESTONE 10.2: L22H05 INTRA-LAYER SPECIFICITY AUDIT (N=32 Tasks x 8 Heads in Layer 22)
✅ Extracted 4D fingerprints and measured 8 Layer-22 heads across all 32 episodes.

📊 INTRA-LAYER PREDICTABILITY SPECTRUM (Layer 22 Heads)
Target Head        | Phi MAE    | Phi ρ      | Global Mean MAE  | Perm p-val   | FDR Sig (q < 0.05)
--------------------------------------------------------------------------------------------------------------------------------------------


In [ ]:
"""
m10_2b_residual_correlation_audit.py
=====================================
Milestone 10.2b: Residual Error Covariance & Latent Causal Factor Audit.

Core Protocol:
  1. Computes Ground-Truth Target Correlation Matrix: Corr(Y) across L22 heads.
  2. Computes Model Prediction Correlation Matrix: Corr(Y_hat).
  3. Computes Residual Error Correlation Matrix: Corr(R = Y - Y_hat).
  4. Performs PCA on Ground Truth vs Residuals to test for a Single Shared Latent Factor.
"""

import torch
import numpy as np
import json
from dataclasses import dataclass
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


# ==============================================================================
# 1. MECHANISTIC HARNESS
# ==============================================================================

class FastL22AuditHarness:
    def __init__(self, model_name: str = "Qwen/Qwen2.5-0.5B", device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"🔄 Loading {model_name} on {self.device.upper()} (bfloat16)...")
        self.model = HookedTransformer.from_pretrained(
            model_name, device=self.device, torch_dtype=torch.bfloat16, default_padding_side="left"
        )
        self.model.eval()
        self.l22_heads = [0, 2, 4, 5, 7, 9, 11, 13]

    @torch.no_grad()
    def extract_phi_and_targets(self, prompt: str) -> Tuple[np.ndarray, np.ndarray]:
        tokens = self.model.to_tokens(prompt)
        nom_logits = self.model(tokens)[0, -1, :].detach().cpu().to(torch.float32)

        top_ids = torch.topk(nom_logits, k=2).indices
        t1, t2 = int(top_ids[0].item()), int(top_ids[1].item())
        base_margin = float((nom_logits[t1] - nom_logits[t2]).item())

        # 4D Micro-Fingerprint (Phi)
        def p1_h6(act, hook): act[:, -1, 6, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p1_h6)]):
            d1 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        def p2_h6(act, hook): act[:, -1, 6, :] *= 1.10; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p2_h6)]):
            d2 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        def p3_h1(act, hook): act[:, -1, 1, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p3_h1)]):
            d3 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        def p4_mlp20(act, hook): act[:, -1, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.20.hook_mlp_out", p4_mlp20)]):
            d4 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        phi = np.array([d1, d2, d3, d4], dtype=np.float32)

        # Ground truth delta on all 8 L22 heads
        l22_deltas = []
        for h in self.l22_heads:
            def h_hook(act, hook, head_id=h):
                act[:, -1, head_id, :] *= 0.0
                return act
            with self.model.hooks(fwd_hooks=[("blocks.22.attn.hook_z", h_hook)]):
                m_logits = self.model(tokens)[0, -1, :].detach().cpu().to(torch.float32)
            d = float((m_logits[t1] - m_logits[t2]).item()) - base_margin
            l22_deltas.append(d)

        return phi, np.array(l22_deltas, dtype=np.float32)


# ==============================================================================
# 2. EVALUATION CORPUS (32 BALANCED TASKS)
# ==============================================================================

def get_corpus() -> List[Dict]:
    return [
        {"id": "F01", "prompt": "The capital of France is the city of"},
        {"id": "F02", "prompt": "The capital of Japan is the bustling city of"},
        {"id": "F03", "prompt": "The official capital city of Germany is"},
        {"id": "F04", "prompt": "The historic capital of Italy is the city of"},
        {"id": "F05", "prompt": "The capital of Spain is the city of"},
        {"id": "F06", "prompt": "The capital of England is the city of"},
        {"id": "F07", "prompt": "The capital of Russia is the city of"},
        {"id": "F08", "prompt": "The capital of Egypt is the city of"},
        {"id": "F09", "prompt": "The capital of Greece is the city of"},
        {"id": "F10", "prompt": "The capital of China is the city of"},
        {"id": "F11", "prompt": "The capital of Canada is the city of"},
        {"id": "F12", "prompt": "The capital of Australia is the city of"},
        {"id": "B01", "prompt": "Alice handed a red apple to Bob. The apple was received by"},
        {"id": "B02", "prompt": "David passed the heavy book to Sarah. The book was taken by"},
        {"id": "B03", "prompt": "John sent a private email to Mary. The email reached"},
        {"id": "B04", "prompt": "Michael offered a cup of tea to Emma. The tea was accepted by"},
        {"id": "B05", "prompt": "Tom tossed the baseball to Jack. The ball was caught by"},
        {"id": "B06", "prompt": "Robert wrote a letter to Lisa. The letter was delivered to"},
        {"id": "B07", "prompt": "William awarded a shiny medal to Oliver. The medal went to"},
        {"id": "B08", "prompt": "James threw the yellow frisbee to Lucas. The frisbee was caught by"},
        {"id": "B09", "prompt": "Daniel gave a fresh sandwich to Mason. The sandwich was eaten by"},
        {"id": "B10", "prompt": "Henry presented an award to Ethan. The award was received by"},
        {"id": "B11", "prompt": "Samuel mailed a package to Alexander. The package arrived for"},
        {"id": "B12", "prompt": "George loaned some money to Noah. The money helped"},
        {"id": "D01", "prompt": "While Rome has the Colosseum, the Eiffel Tower stands in"},
        {"id": "D02", "prompt": "Even though Madrid is famous, the capital of Germany is"},
        {"id": "D03", "prompt": "Berlin has monuments, yet the ancient Colosseum stands in"},
        {"id": "D04", "prompt": "Although Tokyo is very large, the ancient Parthenon is located in"},
        {"id": "D05", "prompt": "Despite London being historic, the capital of Spain is definitely"},
        {"id": "D06", "prompt": "Paris is beautiful, but the capital of Italy is definitely"},
        {"id": "D07", "prompt": "Egypt has pyramids, but the Eiffel Tower was built in the city of"},
        {"id": "D08", "prompt": "Although Moscow has the Kremlin, the capital of France is the city of"},
    ]


# ==============================================================================
# 3. MASTER AUDIT CONTROLLER
# ==============================================================================

def run_milestone_10_2b():
    harness = FastL22AuditHarness()
    corpus = get_corpus()
    head_names = [f"L22H{h:02d}" for h in harness.l22_heads]

    print("\n" + "=" * 125)
    print(f"🔬 MILESTONE 10.2B: RESIDUAL COVARIANCE & LATENT CAUSAL FACTOR AUDIT (N={len(corpus)} Tasks)")
    print("=" * 125)

    phi_list, y_list = [], []
    for item in corpus:
        phi, y_vec = harness.extract_phi_and_targets(item["prompt"])
        phi_list.append(phi)
        y_list.append(y_vec)

    X_phi = np.stack(phi_list)
    Y = np.stack(y_list)  # Shape: (N, 8)
    N, n_targets = Y.shape

    # Fit LOO-Ridge and compute predictions and residuals
    Y_hat = np.zeros_like(Y)
    alpha_reg = 0.5
    for i in range(N):
        tr_idx = [j for j in range(N) if j != i]
        X_tr, y_tr = X_phi[tr_idx], Y[tr_idx]
        w = np.linalg.solve(X_tr.T @ X_tr + alpha_reg * np.eye(X_phi.shape[1]), X_tr.T @ y_tr)
        Y_hat[i] = np.dot(X_phi[i], w)

    R = Y - Y_hat  # Residual errors (Shape: N, 8)

    # Focus on the 3 significant heads + 1 anti-correlated control head
    key_indices = [0, 1, 3, 6]  # H00, H02, H05, H11
    key_labels = [head_names[idx] for idx in key_indices]

    Y_key = Y[:, key_indices]
    Y_hat_key = Y_hat[:, key_indices]
    R_key = R[:, key_indices]

    corr_Y = np.corrcoef(Y_key, rowvar=False)
    corr_Y_hat = np.corrcoef(Y_hat_key, rowvar=False)
    corr_R = np.corrcoef(R_key, rowvar=False)

    # -------------------------------------------------------------------------
    # PRINT CORRELATION MATRICES
    # -------------------------------------------------------------------------
    def print_matrix(title: str, mat: np.ndarray, labels: List[str]):
        print(f"\n📊 {title}")
        header = " " * 12 + " | ".join([f"{l:>9}" for l in labels])
        print(header)
        print("-" * len(header))
        for idx, row_lbl in enumerate(labels):
            row_str = " | ".join([f"{mat[idx, j]:+9.4f}" for j in range(len(labels))])
            print(f"{row_lbl:<12} {row_str}")

    print_matrix("1. GROUND-TRUTH TARGET CORRELATION: Corr(Y) [Actual System Behavior]", corr_Y, key_labels)
    print_matrix("2. MODEL PREDICTION CORRELATION: Corr(Y_hat) [What Phi Captures]", corr_Y_hat, key_labels)
    print_matrix("3. RESIDUAL ERROR CORRELATION: Corr(R = Y - Y_hat) [Unmodeled Coupling]", corr_R, key_labels)

    # -------------------------------------------------------------------------
    # LATENT FACTOR / PCA SPECTRUM
    # -------------------------------------------------------------------------
    print("\n" + "=" * 125)
    print("📊 LATENT FACTOR STRUCTURE: PCA VARIANCE EXPLAINED ON THE 3 COOPERATIVE HEADS (H00, H02, H05)")
    print("=" * 125)

    sub_indices = [0, 1, 3]  # H00, H02, H05
    Y_sub = Y[:, sub_indices]
    R_sub = R[:, sub_indices]

    cov_Y = np.cov(Y_sub, rowvar=False)
    evals_Y, _ = np.linalg.eigh(cov_Y)
    evals_Y = np.sort(evals_Y)[::-1]
    var_exp_Y = (evals_Y / np.sum(evals_Y)) * 100.0

    cov_R = np.cov(R_sub, rowvar=False)
    evals_R, _ = np.linalg.eigh(cov_R)
    evals_R = np.sort(evals_R)[::-1]
    var_exp_R = (evals_R / np.sum(evals_R)) * 100.0

    print(f"  • Ground-Truth Y  : PC1 explains {var_exp_Y[0]:.1f}% | PC2: {var_exp_Y[1]:.1f}% | PC3: {var_exp_Y[2]:.1f}%")
    print(f"  • Residuals R     : PC1 explains {var_exp_R[0]:.1f}% | PC2: {var_exp_R[1]:.1f}% | PC3: {var_exp_R[2]:.1f}%")
    print("-" * 125)

    # Specific head pair correlations in residuals
    r_00_05 = corr_R[0, 2]  # Corr(R_H00, R_H05)
    r_02_05 = corr_R[1, 2]  # Corr(R_H02, R_H05)
    r_00_02 = corr_R[0, 1]  # Corr(R_H00, R_H02)

    mean_res_corr = float(np.mean([abs(r_00_05), abs(r_02_05), abs(r_00_02)]))

    print(f"  • Residual Correlation (H00 vs H05) : {r_00_05:+.4f}")
    print(f"  • Residual Correlation (H02 vs H05) : {r_02_05:+.4f}")
    print(f"  • Residual Correlation (H00 vs H02) : {r_00_02:+.4f}")
    print(f"  • Mean Absolute Residual Corr (H0,H2,H5) : {mean_res_corr:.4f}")
    print("-" * 125)

    # Formal Falsifiable Verdict
    if mean_res_corr >= 0.35 and var_exp_R[0] >= 55.0:
        print("👉 VERDICT: 🟢 STRONG_COUPLED_SUBCIRCUIT_CONFIRMED")
        print("   (H00, H02, H05 share correlated residual errors; they form an integrated downstream functional circuit)")
    elif corr_Y[0, 2] > 0.40 and mean_res_corr < 0.25:
        print("👉 VERDICT: 🟢 PHI_IS_COMMON_EXPLANATORY_SOURCE")
        print("   (Ground-truth outputs are correlated, but residuals are nearly uncorrelated; Phi fully explains their shared variance)")
    else:
        print("👉 VERDICT: 🟡 INDEPENDENT_DOWNSTREAM_PROJECTIONS")
        print("   (Heads exhibit separate, uncorrelated relationships with the base circuit)")
    print("=" * 125 + "\n")

    with open("m10_2b_residual_audit.json", "w", encoding="utf-8") as f:
        json.dump({
            "key_heads": key_labels,
            "corr_ground_truth": corr_Y.tolist(),
            "corr_predictions": corr_Y_hat.tolist(),
            "corr_residuals": corr_R.tolist(),
            "pca_var_explained_Y": var_exp_Y.tolist(),
            "pca_var_explained_R": var_exp_R.tolist(),
            "mean_abs_residual_corr": mean_res_corr
        }, f, indent=2)
    print("📁 Comprehensive Residual Telemetry saved to: m10_2b_residual_audit.json")


if __name__ == "__main__":
    run_milestone_10_2b()

🔄 Loading Qwen/Qwen2.5-0.5B on CUDA (bfloat16)...


/tmp/ipykernel_1228/1354333993.py:33: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer

🔬 MILESTONE 10.2B: RESIDUAL COVARIANCE & LATENT CAUSAL FACTOR AUDIT (N=32 Tasks)

📊 1. GROUND-TRUTH TARGET CORRELATION: Corr(Y) [Actual System Behavior]
               L22H00 |    L22H02 |    L22H05 |    L22H11
---------------------------------------------------------
L22H00         +1.0000 |   +0.1973 |   +0.4865 |   -0.1687
L22H02         +0.1973 |   +1.0000 |   +0.1459 |   +0.1764
L22H05         +0.4865 |   +0.1459 |   +1.0000 |   -0.2447
L22H11         -0.1687 |   +0.1764 |   -0.2447 |   +1.0000

📊 2. MODEL PREDICTION CORRELATION: Corr(Y_hat) [What Phi Captures]
               L22H00 |    L22H02 |    L22H05 |    L22H11
---------------------------------------------------------
L22H00         +1.0000 |   +0.8542 |   +0.9784 |   -0.4321
L22H02         +0.8542 |   +1.0000 |   +0.8690 |   -0.2856
L22H05         +0.9784 |   +0.8690 |   +1.0000 |   -0.3626
L22H11         -0.4321 |   -0.2856 |   -0.3626 |   +1.0000

📊 3. RES

In [ ]:
"""
m10_2c_joint_intervention_epistasis.py
=======================================
Milestone 10.2c: Joint Downstream Intervention & Causal Epistasis Audit.

Hypothesis:
  1. If L22H00 and L22H05 form an integrated downstream functional circuit,
     their joint ablation will exhibit significant causal non-linearity (Epistasis != 0).
  2. The baseline fingerprint Phi(x) can predict the joint consequence Delta(H00 + H05).
"""

import torch
import numpy as np
import json
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


# ==============================================================================
# 1. MECHANISTIC HARNESS
# ==============================================================================

class JointAblationHarness:
    def __init__(self, model_name: str = "Qwen/Qwen2.5-0.5B", device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"🔄 Loading {model_name} on {self.device.upper()} (bfloat16)...")
        self.model = HookedTransformer.from_pretrained(
            model_name, device=self.device, torch_dtype=torch.bfloat16, default_padding_side="left"
        )
        self.model.eval()

    @torch.no_grad()
    def measure_joint_effects(self, prompt: str) -> Tuple[np.ndarray, Dict[str, float]]:
        tokens = self.model.to_tokens(prompt)
        nom_logits = self.model(tokens)[0, -1, :].detach().cpu().to(torch.float32)

        top_ids = torch.topk(nom_logits, k=2).indices
        t1, t2 = int(top_ids[0].item()), int(top_ids[1].item())
        base_margin = float((nom_logits[t1] - nom_logits[t2]).item())

        # 4D Micro-Fingerprint (Phi)
        def p1_h6(act, hook): act[:, -1, 6, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p1_h6)]):
            d1 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        def p2_h6(act, hook): act[:, -1, 6, :] *= 1.10; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p2_h6)]):
            d2 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        def p3_h1(act, hook): act[:, -1, 1, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p3_h1)]):
            d3 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        def p4_mlp20(act, hook): act[:, -1, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.20.hook_mlp_out", p4_mlp20)]):
            d4 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        phi = np.array([d1, d2, d3, d4], dtype=np.float32)

        # 1. Single Ablation L22H00
        def h0_hook(act, hook): act[:, -1, 0, :] *= 0.0; return act
        with self.model.hooks(fwd_hooks=[("blocks.22.attn.hook_z", h0_hook)]):
            d_h0 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        # 2. Single Ablation L22H05
        def h5_hook(act, hook): act[:, -1, 5, :] *= 0.0; return act
        with self.model.hooks(fwd_hooks=[("blocks.22.attn.hook_z", h5_hook)]):
            d_h5 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        # 3. Single Ablation Control L22H02
        def h2_hook(act, hook): act[:, -1, 2, :] *= 0.0; return act
        with self.model.hooks(fwd_hooks=[("blocks.22.attn.hook_z", h2_hook)]):
            d_h2 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        # 4. Joint Ablation: L22H00 + L22H05 (Coupled Pair)
        def joint_05_hook(act, hook):
            act[:, -1, 0, :] *= 0.0
            act[:, -1, 5, :] *= 0.0
            return act
        with self.model.hooks(fwd_hooks=[("blocks.22.attn.hook_z", joint_05_hook)]):
            d_joint_05 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        # 5. Joint Ablation: L22H02 + L22H05 (Control Uncoupled Pair)
        def joint_25_hook(act, hook):
            act[:, -1, 2, :] *= 0.0
            act[:, -1, 5, :] *= 0.0
            return act
        with self.model.hooks(fwd_hooks=[("blocks.22.attn.hook_z", joint_25_hook)]):
            d_joint_25 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        outcomes = {
            "d_h0": d_h0,
            "d_h5": d_h5,
            "d_h2": d_h2,
            "d_joint_05": d_joint_05,
            "d_joint_25": d_joint_25,
            "epistasis_05": d_joint_05 - (d_h0 + d_h5),
            "epistasis_25": d_joint_25 - (d_h2 + d_h5),
        }
        return phi, outcomes


# ==============================================================================
# 2. CORPUS (32 BALANCED TASKS)
# ==============================================================================

def get_corpus() -> List[Dict]:
    return [
        {"id": "F01", "prompt": "The capital of France is the city of"},
        {"id": "F02", "prompt": "The capital of Japan is the bustling city of"},
        {"id": "F03", "prompt": "The official capital city of Germany is"},
        {"id": "F04", "prompt": "The historic capital of Italy is the city of"},
        {"id": "F05", "prompt": "The capital of Spain is the city of"},
        {"id": "F06", "prompt": "The capital of England is the city of"},
        {"id": "F07", "prompt": "The capital of Russia is the city of"},
        {"id": "F08", "prompt": "The capital of Egypt is the city of"},
        {"id": "F09", "prompt": "The capital of Greece is the city of"},
        {"id": "F10", "prompt": "The capital of China is the city of"},
        {"id": "F11", "prompt": "The capital of Canada is the city of"},
        {"id": "F12", "prompt": "The capital of Australia is the city of"},
        {"id": "B01", "prompt": "Alice handed a red apple to Bob. The apple was received by"},
        {"id": "B02", "prompt": "David passed the heavy book to Sarah. The book was taken by"},
        {"id": "B03", "prompt": "John sent a private email to Mary. The email reached"},
        {"id": "B04", "prompt": "Michael offered a cup of tea to Emma. The tea was accepted by"},
        {"id": "B05", "prompt": "Tom tossed the baseball to Jack. The ball was caught by"},
        {"id": "B06", "prompt": "Robert wrote a letter to Lisa. The letter was delivered to"},
        {"id": "B07", "prompt": "William awarded a shiny medal to Oliver. The medal went to"},
        {"id": "B08", "prompt": "James threw the yellow frisbee to Lucas. The frisbee was caught by"},
        {"id": "B09", "prompt": "Daniel gave a fresh sandwich to Mason. The sandwich was eaten by"},
        {"id": "B10", "prompt": "Henry presented an award to Ethan. The award was received by"},
        {"id": "B11", "prompt": "Samuel mailed a package to Alexander. The package arrived for"},
        {"id": "B12", "prompt": "George loaned some money to Noah. The money helped"},
        {"id": "D01", "prompt": "While Rome has the Colosseum, the Eiffel Tower stands in"},
        {"id": "D02", "prompt": "Even though Madrid is famous, the capital of Germany is"},
        {"id": "D03", "prompt": "Berlin has monuments, yet the ancient Colosseum stands in"},
        {"id": "D04", "prompt": "Although Tokyo is very large, the ancient Parthenon is located in"},
        {"id": "D05", "prompt": "Despite London being historic, the capital of Spain is definitely"},
        {"id": "D06", "prompt": "Paris is beautiful, but the capital of Italy is definitely"},
        {"id": "D07", "prompt": "Egypt has pyramids, but the Eiffel Tower was built in the city of"},
        {"id": "D08", "prompt": "Although Moscow has the Kremlin, the capital of France is the city of"},
    ]


# ==============================================================================
# 3. STATISTICAL EVALUATION
# ==============================================================================

def leave_one_out_ridge(X: np.ndarray, y: np.ndarray, alpha: float = 0.5) -> Tuple[float, float]:
    N, D = X.shape
    preds = []
    for i in range(N):
        tr_idx = [j for j in range(N) if j != i]
        X_tr, y_tr = X[tr_idx], y[tr_idx]
        w = np.linalg.solve(X_tr.T @ X_tr + alpha * np.eye(D), X_tr.T @ y_tr)
        preds.append(float(np.dot(X[i], w)))
    preds = np.array(preds)
    mae = float(np.mean(np.abs(preds - y)))
    corr = float(np.corrcoef(preds, y)[0, 1]) if np.std(preds) > 1e-6 else 0.0
    return mae, corr


# ==============================================================================
# 4. MASTER RUNNER
# ==============================================================================

def run_milestone_10_2c():
    harness = JointAblationHarness()
    corpus = get_corpus()

    print("\n" + "=" * 135)
    print(f"🔬 MILESTONE 10.2C: JOINT DOWNSTREAM ABLATION & CAUSAL EPISTASIS AUDIT (N={len(corpus)} Tasks)")
    print("=" * 135)

    phi_list = []
    records = {k: [] for k in ["d_h0", "d_h5", "d_h2", "d_joint_05", "d_joint_25", "epistasis_05", "epistasis_25"]}

    for item in corpus:
        phi, outcomes = harness.measure_joint_effects(item["prompt"])
        phi_list.append(phi)
        for k in records:
            records[k].append(outcomes[k])

    X_phi = np.stack(phi_list)
    for k in records:
        records[k] = np.array(records[k])

    # 1. Evaluate Predictability of Joint Ablation from Phi
    mae_joint_05, corr_joint_05 = leave_one_out_ridge(X_phi, records["d_joint_05"])
    mae_joint_25, corr_joint_25 = leave_one_out_ridge(X_phi, records["d_joint_25"])

    # 2. Epistasis Quantiles & Magnitude
    mean_epi_05 = float(np.mean(np.abs(records["epistasis_05"])))
    mean_epi_25 = float(np.mean(np.abs(records["epistasis_25"])))

    mean_joint_05 = float(np.mean(np.abs(records["d_joint_05"])))
    epi_ratio_05 = (mean_epi_05 / (mean_joint_05 + 1e-8)) * 100.0

    print("📊 1. PREDICTABILITY OF JOINT INTERVENTIONS FROM BASE-CIRCUIT PHI")
    print("-" * 135)
    print(f"  • Joint Ablation (H00 + H05) [Coupled Pair]   : Phi MAE = {mae_joint_05:.4f} | Pearson ρ = {corr_joint_05:+.4f}")
    print(f"  • Joint Ablation (H02 + H05) [Uncoupled Pair] : Phi MAE = {mae_joint_25:.4f} | Pearson ρ = {corr_joint_25:+.4f}")

    print("\n📊 2. CAUSAL EPISTASIS & NON-LINEARITY ANALYSIS: Δ(Joint) - (ΔA + ΔB)")
    print("-" * 135)
    print(f"  • Mean Absolute Epistasis (H00 + H05) : {mean_epi_05:.4f}  ({epi_ratio_05:.1f}% of total joint effect)")
    print(f"  • Mean Absolute Epistasis (H02 + H05) : {mean_epi_25:.4f}")
    print("-" * 135)

    if corr_joint_05 >= 0.50 and mean_epi_05 > 0.05:
        print("👉 VERDICT: 🟢 INTEGRATED_FUNCTIONAL_CIRCUIT_CONFIRMED")
        print("   (L22H00 and L22H05 exhibit non-linear causal interaction and are jointly predictable from Phi)")
    elif corr_joint_05 >= 0.50:
        print("👉 VERDICT: 🟢 PARALLEL_ADDITIVE_COUPLING")
        print("   (L22H00 and L22H05 share common predictive structure with linear superposition)")
    else:
        print("👉 VERDICT: 🔴 WEAK_JOINT_PREDICTABILITY")
    print("=" * 135 + "\n")

    with open("m10_2c_joint_results.json", "w", encoding="utf-8") as f:
        json.dump({
            "corr_joint_05": corr_joint_05,
            "corr_joint_25": corr_joint_25,
            "mean_abs_epistasis_05": mean_epi_05,
            "mean_abs_epistasis_25": mean_epi_25,
            "epistasis_ratio_05_pct": epi_ratio_05
        }, f, indent=2)
    print("📁 Joint Telemetry saved to: m10_2c_joint_results.json")


if __name__ == "__main__":
    run_milestone_10_2c()

In [ ]:
"""
m10_2_null_prediction_permutation_audit.py
===========================================
Diagnostic Audit: Distinguishing Real Latent Coupling from Low-Rank Ridge Artifacts.

Test:
  Shuffles target outputs Y independently per component (destroying true causal signals)
  and measures whether Corr(Y_hat_i, Y_hat_j) remains artificially high purely due to
  the dominant subspace of Phi.
"""

import numpy as np
import json
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

def run_prediction_permutation_audit(data_file: str = "m10_2b_residual_audit.json"):
    # Load previously computed telemetry or generate diagnostic arrays
    try:
        with open("m10_2b_residual_audit.json", "r") as f:
            prev_data = json.load(f)
            print("✅ Loaded telemetry from m10_2b_residual_audit.json")
    except FileNotFoundError:
        print("⚠️ File not found. Run this in the same directory as Milestone 10.2b results.")

    # Synthetic reproduction of typical Phi / Y structures if running standalone
    # Or load actual arrays if available in runtime
    pass

def execute_null_check(X_phi: np.ndarray, Y_targets: np.ndarray, labels: list, n_perms: int = 1000, alpha_reg: float = 0.5):
    """
    X_phi: (N, 4)
    Y_targets: (N, K) -> e.g. H00, H02, H05, H11
    """
    N, D = X_phi.shape
    K = Y_targets.shape[1]

    def fit_loo_predictions(X, Y):
        Y_preds = np.zeros_like(Y)
        for i in range(N):
            tr_idx = [j for j in range(N) if j != i]
            X_tr, y_tr = X[tr_idx], Y[tr_idx]
            w = np.linalg.solve(X_tr.T @ X_tr + alpha_reg * np.eye(D), X_tr.T @ y_tr)
            Y_preds[i] = np.dot(X[i], w)
        return Y_preds

    # 1. Real Model Prediction Correlation
    Y_hat_real = fit_loo_predictions(X_phi, Y_targets)
    corr_real = np.corrcoef(Y_hat_real, rowvar=False)

    # 2. Permutation Null Test
    null_corrs = {f"{labels[i]}-{labels[j]}": [] for i in range(K) for j in range(i + 1, K)}

    for _ in range(n_perms):
        # Shuffle each target Y column independently to break true cross-component alignment
        Y_shuff = np.zeros_like(Y_targets)
        for k in range(K):
            Y_shuff[:, k] = np.random.permutation(Y_targets[:, k])

        Y_hat_shuff = fit_loo_predictions(X_phi, Y_shuff)
        corr_shuff = np.corrcoef(Y_hat_shuff, rowvar=False)

        for i in range(K):
            for j in range(i + 1, K):
                null_corrs[f"{labels[i]}-{labels[j]}"].append(corr_shuff[i, j])

    # 3. Statistical Reporting
    print("\n" + "=" * 120)
    print(f"{'Component Pair':<20} | {'Real Corr(Y_hat)':<18} | {'Null Mean (Artifact)':<22} | {'Null 95th Percentile':<22} | {'Verdict'}")
    print("-" * 120)

    results_summary = {}
    for i in range(K):
        for j in range(i + 1, K):
            pair_key = f"{labels[i]}-{labels[j]}"
            r_val = corr_real[i, j]
            null_dist = np.array(null_corrs[pair_key])
            null_mean = float(np.mean(null_dist))
            null_p95 = float(np.percentile(null_dist, 95))
            p_val = float((np.sum(null_dist >= r_val) + 1) / (n_perms + 1))

            if null_mean > 0.60:
                verdict = "🔴 GEOMETRIC ARTIFACT (Dominant Phi Subspace)"
            elif r_val > null_p95 and p_val < 0.05:
                verdict = "🟢 REAL ALIGNMENT (Beyond Geometry)"
            else:
                verdict = "🟡 INDETERMINATE / WEAK"

            print(f"{pair_key:<20} | {r_val:+16.4f} | {null_mean:+20.4f} | {null_p95:+20.4f} | {verdict}")

            results_summary[pair_key] = {
                "real_corr_y_hat": r_val,
                "null_mean": null_mean,
                "null_p95": null_p95,
                "p_value": p_val,
                "verdict": verdict
            }

    print("=" * 120 + "\n")
    return results_summary

In [ ]:
"""
m10_2_null_prediction_permutation_audit_complete.py
====================================================
Diagnostic Audit: Real Latent Signal vs. Low-Rank Ridge Geometric Artifact.
"""

import torch
import numpy as np
import json
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


# ==============================================================================
# 1. MECHANISTIC HARNESS & DATA EXTRACTION
# ==============================================================================

class FastAuditHarness:
    def __init__(self, model_name: str = "Qwen/Qwen2.5-0.5B", device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"🔄 Loading {model_name} on {self.device.upper()} (bfloat16)...")
        self.model = HookedTransformer.from_pretrained(
            model_name, device=self.device, torch_dtype=torch.bfloat16, default_padding_side="left"
        )
        self.model.eval()
        self.target_heads = [0, 2, 5, 11]  # H00, H02, H05, H11

    @torch.no_grad()
    def extract_phi_and_targets(self, prompt: str) -> Tuple[np.ndarray, np.ndarray]:
        tokens = self.model.to_tokens(prompt)
        nom_logits = self.model(tokens)[0, -1, :].detach().cpu().to(torch.float32)

        top_ids = torch.topk(nom_logits, k=2).indices
        t1, t2 = int(top_ids[0].item()), int(top_ids[1].item())
        base_margin = float((nom_logits[t1] - nom_logits[t2]).item())

        # 4D Micro-Fingerprint (Phi)
        def p1_h6(act, hook): act[:, -1, 6, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p1_h6)]):
            d1 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        def p2_h6(act, hook): act[:, -1, 6, :] *= 1.10; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p2_h6)]):
            d2 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        def p3_h1(act, hook): act[:, -1, 1, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.21.attn.hook_z", p3_h1)]):
            d3 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        def p4_mlp20(act, hook): act[:, -1, :] *= 0.90; return act
        with self.model.hooks(fwd_hooks=[("blocks.20.hook_mlp_out", p4_mlp20)]):
            d4 = float((self.model(tokens)[0, -1, t1] - self.model(tokens)[0, -1, t2]).item()) - base_margin

        phi = np.array([d1, d2, d3, d4], dtype=np.float32)

        # Measure 4 Target Heads in Layer 22
        deltas = []
        for h in self.target_heads:
            def h_hook(act, hook, head_id=h):
                act[:, -1, head_id, :] *= 0.0
                return act
            with self.model.hooks(fwd_hooks=[("blocks.22.attn.hook_z", h_hook)]):
                m_logits = self.model(tokens)[0, -1, :].detach().cpu().to(torch.float32)
            d = float((m_logits[t1] - m_logits[t2]).item()) - base_margin
            deltas.append(d)

        return phi, np.array(deltas, dtype=np.float32)


# ==============================================================================
# 2. CORPUS (32 BALANCED TASKS)
# ==============================================================================

def get_corpus() -> List[Dict]:
    return [
        {"id": "F01", "prompt": "The capital of France is the city of"},
        {"id": "F02", "prompt": "The capital of Japan is the bustling city of"},
        {"id": "F03", "prompt": "The official capital city of Germany is"},
        {"id": "F04", "prompt": "The historic capital of Italy is the city of"},
        {"id": "F05", "prompt": "The capital of Spain is the city of"},
        {"id": "F06", "prompt": "The capital of England is the city of"},
        {"id": "F07", "prompt": "The capital of Russia is the city of"},
        {"id": "F08", "prompt": "The capital of Egypt is the city of"},
        {"id": "F09", "prompt": "The capital of Greece is the city of"},
        {"id": "F10", "prompt": "The capital of China is the city of"},
        {"id": "F11", "prompt": "The capital of Canada is the city of"},
        {"id": "F12", "prompt": "The capital of Australia is the city of"},
        {"id": "B01", "prompt": "Alice handed a red apple to Bob. The apple was received by"},
        {"id": "B02", "prompt": "David passed the heavy book to Sarah. The book was taken by"},
        {"id": "B03", "prompt": "John sent a private email to Mary. The email reached"},
        {"id": "B04", "prompt": "Michael offered a cup of tea to Emma. The tea was accepted by"},
        {"id": "B05", "prompt": "Tom tossed the baseball to Jack. The ball was caught by"},
        {"id": "B06", "prompt": "Robert wrote a letter to Lisa. The letter was delivered to"},
        {"id": "B07", "prompt": "William awarded a shiny medal to Oliver. The medal went to"},
        {"id": "B08", "prompt": "James threw the yellow frisbee to Lucas. The frisbee was caught by"},
        {"id": "B09", "prompt": "Daniel gave a fresh sandwich to Mason. The sandwich was eaten by"},
        {"id": "B10", "prompt": "Henry presented an award to Ethan. The award was received by"},
        {"id": "B11", "prompt": "Samuel mailed a package to Alexander. The package arrived for"},
        {"id": "B12", "prompt": "George loaned some money to Noah. The money helped"},
        {"id": "D01", "prompt": "While Rome has the Colosseum, the Eiffel Tower stands in"},
        {"id": "D02", "prompt": "Even though Madrid is famous, the capital of Germany is"},
        {"id": "D03", "prompt": "Berlin has monuments, yet the ancient Colosseum stands in"},
        {"id": "D04", "prompt": "Although Tokyo is very large, the ancient Parthenon is located in"},
        {"id": "D05", "prompt": "Despite London being historic, the capital of Spain is definitely"},
        {"id": "D06", "prompt": "Paris is beautiful, but the capital of Italy is definitely"},
        {"id": "D07", "prompt": "Egypt has pyramids, but the Eiffel Tower was built in the city of"},
        {"id": "D08", "prompt": "Although Moscow has the Kremlin, the capital of France is the city of"},
    ]


# ==============================================================================
# 3. NULL PERMUTATION AUDIT ENGINE
# ==============================================================================

def fit_loo_predictions(X: np.ndarray, Y: np.ndarray, alpha_reg: float = 0.5) -> np.ndarray:
    N, D = X.shape
    Y_preds = np.zeros_like(Y)
    for i in range(N):
        tr_idx = [j for j in range(N) if j != i]
        X_tr, y_tr = X[tr_idx], Y[tr_idx]
        w = np.linalg.solve(X_tr.T @ X_tr + alpha_reg * np.eye(D), X_tr.T @ y_tr)
        Y_preds[i] = np.dot(X[i], w)
    return Y_preds


def run_full_permutation_audit():
    harness = FastAuditHarness()
    corpus = get_corpus()
    labels = ["L22H00", "L22H02", "L22H05", "L22H11"]

    print("\n" + "=" * 125)
    print(f"🔬 RUNNING 1000-ITERATION NULL PERMUTATION AUDIT (N={len(corpus)} Tasks)")
    print("=" * 125)

    phi_list, y_list = [], []
    for item in corpus:
        phi, y_vec = harness.extract_phi_and_targets(item["prompt"])
        phi_list.append(phi)
        y_list.append(y_vec)

    X_phi = np.stack(phi_list)
    Y_targets = np.stack(y_list)
    N, K = Y_targets.shape
    n_perms = 1000

    # 1. Real Model Predictions and Correlation Matrix
    Y_hat_real = fit_loo_predictions(X_phi, Y_targets)
    corr_real = np.corrcoef(Y_hat_real, rowvar=False)

    # 2. Permutation Null Test
    null_corrs = {f"{labels[i]}-{labels[j]}": [] for i in range(K) for j in range(i + 1, K)}

    for _ in range(n_perms):
        # Shuffle each target column independently
        Y_shuff = np.zeros_like(Y_targets)
        for k in range(K):
            Y_shuff[:, k] = np.random.permutation(Y_targets[:, k])

        Y_hat_shuff = fit_loo_predictions(X_phi, Y_shuff)
        corr_shuff = np.corrcoef(Y_hat_shuff, rowvar=False)

        for i in range(K):
            for j in range(i + 1, K):
                null_corrs[f"{labels[i]}-{labels[j]}"].append(corr_shuff[i, j])

    # 3. Print Results Table
    print("\n" + "=" * 130)
    print(f"{'Component Pair':<18} | {'Real Corr(Ŷ)':<14} | {'Null Mean (Artifact)':<22} | {'Null 95th Percentile':<22} | {'p-value':<10} | {'Verdict'}")
    print("-" * 130)

    summary_records = {}
    for i in range(K):
        for j in range(i + 1, K):
            pair_key = f"{labels[i]}-{labels[j]}"
            r_val = float(corr_real[i, j])
            null_dist = np.array(null_corrs[pair_key])
            null_mean = float(np.mean(null_dist))
            null_p95 = float(np.percentile(null_dist, 95))
            p_val = float((np.sum(null_dist >= r_val) + 1) / (n_perms + 1))

            if null_mean >= 0.60:
                verdict = "🔴 GEOMETRIC ARTIFACT (Dominant Phi Subspace)"
            elif r_val >= null_p95 and p_val < 0.05:
                verdict = "🟢 REAL SIGNAL (Beyond Geometry)"
            else:
                verdict = "🟡 NO EFFECT / WEAK"

            print(f"{pair_key:<18} | {r_val:+12.4f}  | {null_mean:+20.4f} | {null_p95:+20.4f} | p={p_val:.4f}  | {verdict}")

            summary_records[pair_key] = {
                "real_corr": r_val,
                "null_mean": null_mean,
                "null_p95": null_p95,
                "p_value": p_val,
                "verdict": verdict
            }

    print("=" * 130 + "\n")

    with open("m10_2_null_audit_results.json", "w", encoding="utf-8") as f:
        json.dump(summary_records, f, indent=2)
    print("📁 Telemetry saved to: m10_2_null_audit_results.json")


# Direct execution entry point
if __name__ == "__main__":
    run_full_permutation_audit()

In [ ]:
"""
m10_2d_epistasis_random_controls_audit.py
==========================================
Milestone 10.2d: Epistasis Random Control Baseline & Specificity Z-Score.

Evaluates whether the 48.0% causal non-linearity (epistasis) observed in (H00, H05)
is a privileged circuit-level interaction or a generic artifact of dual-head ablation in Layer 22.
"""

import torch
import numpy as np
import json
from dataclasses import dataclass
from typing import Dict, List, Tuple
from transformer_lens import HookedTransformer

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


# ==============================================================================
# 1. MECHANISTIC HARNESS
# ==============================================================================

class MultiPairEpistasisHarness:
    def __init__(self, model_name: str = "Qwen/Qwen2.5-0.5B", device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"🔄 Loading {model_name} on {self.device.upper()} (bfloat16)...")
        self.model = HookedTransformer.from_pretrained(
            model_name, device=self.device, torch_dtype=torch.bfloat16, default_padding_side="left"
        )
        self.model.eval()

    @torch.no_grad()
    def measure_pair_epistasis(self, prompt: str, pairs: List[Tuple[int, int]]) -> Dict[str, Dict[str, float]]:
        tokens = self.model.to_tokens(prompt)
        nom_logits = self.model(tokens)[0, -1, :].detach().cpu().to(torch.float32)

        top_ids = torch.topk(nom_logits, k=2).indices
        t1, t2 = int(top_ids[0].item()), int(top_ids[1].item())
        base_margin = float((nom_logits[t1] - nom_logits[t2]).item())

        # Collect unique individual heads needed
        unique_heads = sorted(list(set([h for p in pairs for h in p])))
        single_deltas = {}

        for h in unique_heads:
            def single_hook(act, hook, head_id=h):
                act[:, -1, head_id, :] *= 0.0
                return act
            with self.model.hooks(fwd_hooks=[("blocks.22.attn.hook_z", single_hook)]):
                m_logits = self.model(tokens)[0, -1, :].detach().cpu().to(torch.float32)
            single_deltas[h] = float((m_logits[t1] - m_logits[t2]).item()) - base_margin

        # Measure joint ablation for each pair
        pair_results = {}
        for h_a, h_b in pairs:
            pair_key = f"L22H{h_a:02d}+L22H{h_b:02d}"

            def joint_hook(act, hook, a=h_a, b=h_b):
                act[:, -1, a, :] *= 0.0
                act[:, -1, b, :] *= 0.0
                return act

            with self.model.hooks(fwd_hooks=[("blocks.22.attn.hook_z", joint_hook)]):
                m_logits_joint = self.model(tokens)[0, -1, :].detach().cpu().to(torch.float32)

            d_joint = float((m_logits_joint[t1] - m_logits_joint[t2]).item()) - base_margin
            d_sum = single_deltas[h_a] + single_deltas[h_b]
            epistasis = d_joint - d_sum

            pair_results[pair_key] = {
                "d_joint": d_joint,
                "d_sum": d_sum,
                "epistasis": epistasis,
            }

        return pair_results


# ==============================================================================
# 2. CORPUS (32 BALANCED TASKS)
# ==============================================================================

def get_corpus() -> List[Dict]:
    return [
        {"id": "F01", "prompt": "The capital of France is the city of"},
        {"id": "F02", "prompt": "The capital of Japan is the bustling city of"},
        {"id": "F03", "prompt": "The official capital city of Germany is"},
        {"id": "F04", "prompt": "The historic capital of Italy is the city of"},
        {"id": "F05", "prompt": "The capital of Spain is the city of"},
        {"id": "F06", "prompt": "The capital of England is the city of"},
        {"id": "F07", "prompt": "The capital of Russia is the city of"},
        {"id": "F08", "prompt": "The capital of Egypt is the city of"},
        {"id": "F09", "prompt": "The capital of Greece is the city of"},
        {"id": "F10", "prompt": "The capital of China is the city of"},
        {"id": "F11", "prompt": "The capital of Canada is the city of"},
        {"id": "F12", "prompt": "The capital of Australia is the city of"},
        {"id": "B01", "prompt": "Alice handed a red apple to Bob. The apple was received by"},
        {"id": "B02", "prompt": "David passed the heavy book to Sarah. The book was taken by"},
        {"id": "B03", "prompt": "John sent a private email to Mary. The email reached"},
        {"id": "B04", "prompt": "Michael offered a cup of tea to Emma. The tea was accepted by"},
        {"id": "B05", "prompt": "Tom tossed the baseball to Jack. The ball was caught by"},
        {"id": "B06", "prompt": "Robert wrote a letter to Lisa. The letter was delivered to"},
        {"id": "B07", "prompt": "William awarded a shiny medal to Oliver. The medal went to"},
        {"id": "B08", "prompt": "James threw the yellow frisbee to Lucas. The frisbee was caught by"},
        {"id": "B09", "prompt": "Daniel gave a fresh sandwich to Mason. The sandwich was eaten by"},
        {"id": "B10", "prompt": "Henry presented an award to Ethan. The award was received by"},
        {"id": "B11", "prompt": "Samuel mailed a package to Alexander. The package arrived for"},
        {"id": "B12", "prompt": "George loaned some money to Noah. The money helped"},
        {"id": "D01", "prompt": "While Rome has the Colosseum, the Eiffel Tower stands in"},
        {"id": "D02", "prompt": "Even though Madrid is famous, the capital of Germany is"},
        {"id": "D03", "prompt": "Berlin has monuments, yet the ancient Colosseum stands in"},
        {"id": "D04", "prompt": "Although Tokyo is very large, the ancient Parthenon is located in"},
        {"id": "D05", "prompt": "Despite London being historic, the capital of Spain is definitely"},
        {"id": "D06", "prompt": "Paris is beautiful, but the capital of Italy is definitely"},
        {"id": "D07", "prompt": "Egypt has pyramids, but the Eiffel Tower was built in the city of"},
        {"id": "D08", "prompt": "Although Moscow has the Kremlin, the capital of France is the city of"},
    ]


# ==============================================================================
# 3. MASTER RUNNER
# ==============================================================================

def run_milestone_10_2d():
    harness = MultiPairEpistasisHarness()
    corpus = get_corpus()

    # Target pair + 6 distinct random/control pairs in Layer 22 (valid indices 0..13)
    target_pair = (0, 5)
    control_pairs = [
        (2, 7),
        (4, 9),
        (1, 11),
        (3, 13),
        (6, 10),
        (2, 5),
    ]
    all_pairs = [target_pair] + control_pairs

    print("\n" + "=" * 135)
    print(f"🔬 MILESTONE 10.2D: EPISTASIS RANDOM CONTROLS AUDIT (N={len(corpus)} Tasks x 7 Pairs in Layer 22)")
    print("=" * 135)

    pair_data = {f"L22H{a:02d}+L22H{b:02d}": {"d_joint": [], "d_sum": [], "epistasis": []} for a, b in all_pairs}

    for item in corpus:
        res = harness.measure_pair_epistasis(item["prompt"], all_pairs)
        for k in res:
            pair_data[k]["d_joint"].append(res[k]["d_joint"])
            pair_data[k]["d_sum"].append(res[k]["d_sum"])
            pair_data[k]["epistasis"].append(res[k]["epistasis"])

    # Compute Epistasis Metrics per Pair
    summary_rows = []
    target_key = f"L22H{target_pair[0]:02d}+L22H{target_pair[1]:02d}"

    for k in pair_data:
        arr_joint = np.array(pair_data[k]["d_joint"])
        arr_sum = np.array(pair_data[k]["d_sum"])
        arr_epi = np.array(pair_data[k]["epistasis"])

        mean_abs_joint = float(np.mean(np.abs(arr_joint)))
        mean_abs_epi = float(np.mean(np.abs(arr_epi)))
        epi_ratio = (mean_abs_epi / (mean_abs_joint + 1e-8)) * 100.0

        # Correlation between linear sum and joint outcome (1.0 = perfect linearity)
        lin_corr = float(np.corrcoef(arr_sum, arr_joint)[0, 1]) if np.std(arr_sum) > 1e-6 else 1.0

        summary_rows.append({
            "pair": k,
            "is_target": (k == target_key),
            "mean_abs_joint": mean_abs_joint,
            "mean_abs_epistasis": mean_abs_epi,
            "epistasis_ratio_pct": epi_ratio,
            "linearity_corr": lin_corr,
        })

    # -------------------------------------------------------------------------
    # PRINT SUMMARY TABLE
    # -------------------------------------------------------------------------
    print(f"{'Pair':<18} | {'Type':<12} | {'Mean |Joint|':<14} | {'Mean |Epistasis|':<18} | {'Epistasis Ratio (%)':<22} | {'Linearity ρ (Sum vs Joint)'}")
    print("-" * 135)

    for row in summary_rows:
        type_str = "TARGET ◄" if row["is_target"] else "CONTROL"
        print(f"{row['pair']:<18} | {type_str:<12} | {row['mean_abs_joint']:12.4f}   | {row['mean_abs_epistasis']:16.4f}   | "
              f"{row['epistasis_ratio_pct']:18.1f}%   | {row['linearity_corr']:+12.4f}")

    # -------------------------------------------------------------------------
    # Z-SCORE & SPECIFICITY METRICS
    # -------------------------------------------------------------------------
    print("\n" + "=" * 135)
    print("📊 EPISTASIS SPECIFICITY & OUTLIER ANALYSIS")
    print("=" * 135)

    control_ratios = [r["epistasis_ratio_pct"] for r in summary_rows if not r["is_target"]]
    target_ratio = next(r["epistasis_ratio_pct"] for r in summary_rows if r["is_target"])

    mean_ctrl_ratio = float(np.mean(control_ratios))
    std_ctrl_ratio = float(np.std(control_ratios)) if np.std(control_ratios) > 1e-6 else 1e-4
    z_score_epi = (target_ratio - mean_ctrl_ratio) / std_ctrl_ratio

    print(f"  • Target Pair (H00+H05) Epistasis Ratio : {target_ratio:.1f}%")
    print(f"  • Control Pairs Mean Epistasis Ratio    : {mean_ctrl_ratio:.1f}% (± {std_ctrl_ratio:.1f}%)")
    print(f"  • Epistasis Specificity Z-Score         : Z = {z_score_epi:+.2f}")
    print("-" * 135)

    if z_score_epi >= 1.50:
        print("👉 FINAL AUDIT VERDICT: 🟢 PRIVILEGED_NONLINEAR_CIRCUIT_VALIDATED")
        print("   (H00+H05 non-linear epistasis significantly exceeds the Layer-22 background baseline)")
    elif target_ratio >= mean_ctrl_ratio:
        print("👉 FINAL AUDIT VERDICT: 🟡 WEAK_OR_MODERATE_NONLINEARITY")
        print("   (Epistasis exists, but is partially comparable to baseline multi-head interference)")
    else:
        print("👉 FINAL AUDIT VERDICT: 🔴 GENERIC_MULTI_HEAD_ARTIFACT")
    print("=" * 135 + "\n")

    with open("m10_2d_epistasis_audit.json", "w", encoding="utf-8") as f:
        json.dump({
            "summary_rows": summary_rows,
            "target_ratio": target_ratio,
            "control_mean_ratio": mean_ctrl_ratio,
            "control_std_ratio": std_ctrl_ratio,
            "z_score": z_score_epi,
        }, f, indent=2)
    print("📁 Comprehensive Epistasis Telemetry saved to: m10_2d_epistasis_audit.json")


if __name__ == "__main__":
    run_milestone_10_2d()